In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`


In [2]:
using LinearAlgebra
using SparseArrays
using Convex
using GLPK, MathOptInterface, JuMP
using Plots
using MosekTools
plotlyjs()

pgfplotsx()

WebIO._IJuliaInit()

┌ Warning: backend `PlotlyJS` is not installed.
└ @ Plots /Users/Josephine/.julia/packages/Plots/nuwp4/src/backends.jl:37
┌ Warning: backend `PGFPlotsX` is not installed.
└ @ Plots /Users/Josephine/.julia/packages/Plots/nuwp4/src/backends.jl:37


Plots.PGFPlotsXBackend()

In [3]:
### Constants
const μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
# const J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)const
const J2 = 1082.63*10^-6 #Earth J2 gravity coefficient (dimensionless) 
const J3 = -2.52*10^-6 #Earth J3 gravity coefficient (dimensionless)
const J4 = -1.61*10^-6 #Earth J4 gravity coefficient (dimensionless)
const J5 = -0.15*10^-6 #Earth J5 gravity coefficient (dimensionless)
const J6 =  0.57*10^-6 #Earth J6 gravity coefficient (dimensionless)
const Re_mks = 6378.137 #Radius of the Earth (km)
const Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

const Cd = 2.2 #Drag coefficient (unitless)
const AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
const AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [4]:
#Orbital Parameters (roughly ISS)
const a_mks = 6791.0 #semi-major axis (km)
const inc = 54.74 #inclination (deg)
const n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
const P_mks = 2*pi/n_mks; #orbital period (sec) 1.54 hour 5569.440609858254 sec


In [5]:
#Normalize units
const a = 1.0 #distance unit is semi-major axis
const P = 1 #time unit is orbital period
const μ = (2*pi/P)^2 * a^3 #μ_mks*(P_mks^2/a_mks^3)
const n = sqrt(μ/a^3)
const Re = Re_mks/a_mks
const Ωe = Ωe_mks*P_mks;

const AoM = AoM_mks/(a_mks^2)
const AoM0 = AoM0_mks/(a_mks^2);


In [6]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    θ = atan(x[2],x[1]) #azimuth
    ϕ = atan(sqrt(x[1]^2+x[2]^2),x[3]) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_θaΩ(x)
    r = x[1:3]
    v = x[4:6]
        
    E = 0.5*v'*v + potential(r)
    
    a = -μ/(2*E)
    
    i_x = [1,0,0]
    i_y = [0,1,0]
    i_z = [0,0,1]
    
    h = cross(r,v)
    Ω = atan(h[2]/norm(h[1:2]), h[1]/norm(h[1:2]))
    
    n = cross(i_z,h)/norm(cross(i_z,h))
    if dot(r,i_z) > 0
        θ = acos(dot(n,(r/norm(r))))
    else
        θ = 2*pi- acos(dot(n,(r/norm(r))))
    end
    return [θ; a; Ω]
end

function cartesian_to_ΔθaΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(r0)
    E1 = 0.5*v1'*v1 + potential(r1)
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    

    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))

    
    h0 = cross(r0,v0)
    Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
    Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    
    ΔΩ = Ω1-Ω0
    
    return [Δθ; Δa; ΔΩ]
end

function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [7]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [8]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1
function kvalues(a)
    n = sqrt(μ/a^3)
    Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2#-15/a_mks
    ȧ = 2*(a^2/μ)*(a)*n*Fd
    k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
    k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
    k3 = ȧ #dȧ/d(Drag) - 3*sqrt(μ*a)*J2*Re*Re/a^3*sind(inc)^2 
    k4 = k1/k2
    return [k1, k2, k3, k4]
end

kvalues (generic function with 1 method)

In [9]:
#Discrete-time model
num_sat = 3 # number of couple of satellites 


function create_continuous(a, num_sat)
    #Discrete-time model
#     k1, k2, k3, k4 = kvalues(a)
    A_bar = zeros(3*num_sat,3*num_sat) 
    B_bar = zeros(3*num_sat,num_sat+1) 

    # B_bar[1:3,1:2]=B0
    for k =0:num_sat-1
        k1, k2, k3, k4 = kvalues(a[k+1])
        A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = [0 k2 0; 0 0 0; 0 k1 0]
        B_bar[3*k+2,1] = -k3
        B_bar[3*k+2,2+k] = k3
    end
    return A_bar, B_bar 
end

function c2d(a,dt,num_sat)
    # discretize a continuous time linear system 
    # convert xdot = Ax + Bu 
    # into x_{k+1} = Ad x_k + Bd x_k 
    # where we have dt between k and k +1 
    
    A,B = create_continuous(a, num_sat)
    nx,nu = size(B)
    
    H = exp(dt*[A B;zeros(nu, nx + nu)])
    Ad = H[1:nx, 1:nx]
    Bd = H[1:nx, (nx + 1):(nx + nu)]
    return Ad, Bd 
end

a_vec = a*ones(num_sat)
const h =60 #time step
Ad, Bd = c2d(a_vec, h, num_sat);


In [10]:
function sat_dynamics_rk4(x,u,h)
    f1 = satellite_dynamics(x,u)
    f2 = satellite_dynamics(x + 0.5*h*f1,u)
    f3 = satellite_dynamics(x + 0.5*h*f2, u)
    f4 = satellite_dynamics(x + h*f3, u)
    return x + (h/6.0)*(f1 + 2*f2 +2*f3 +f4)
end

sat_dynamics_rk4 (generic function with 1 method)

In [11]:
### Full Orbital Dynamics with J2 and Drag
function potential(s)
    #Input is position in spherical coordinates
#     r = s[1] #range
#     ϕ = s[2] #azimuth
#     θ = s[3] #elevation measured up from equatorial plane
    
    #Input is position in cartesian coordinates
    r = norm(s)
    sin_ϕ = s[3]/r
    ϕ = atan(sqrt(s[1]^2+s[2]^2)/s[3]) #azimuth
    
    x = sin_ϕ#cos(ϕ)#
    
    U_J2 = μ/r*J2/2*(Re/r)^2*(3*x^2 - 1.0)
    U_J3 = μ/r*J3/2*(Re/r)^3*(5*x^3 - 3*x)
    U_J4 = μ/r*J4/8*(Re/r)^4*(35*x^4 -30*x^2 + 3)
    U_J5 = μ/r*J5/8*(Re/r)^5*(63*x^5 - 70*x^3 + 15*x)
    U_J6 = μ/r*J6/16*(Re/r)^6*(231*x^6 - 315*x^4 + 105*x^2 - 5)
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + U_J2 #+ U_J3 + U_J4 + U_J5 + U_J6  #(0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

# function to evaluate J and keplerian gravitational acceleration
function gravitational_acceleration(s)
    x = s[1]
    y = s[2]
    z = s[3]
    r = sqrt(s[1]^2+s[2]^2+s[3]^2)
    
    sinϕ = z/r
    g_kp = - μ_mks*s/(r^3)
    g_j2 = -3/2*J2*(μ_mks/r^2)*(Re_mks^2/r^2)*[
        (1-5*sinϕ^2)*x/r;
        (1-5*sinϕ^2)*y/r;
        (3-5*sinϕ^2)*z/r]
    
    g_j3 = -1/2*J3*(μ_mks/r^2)*(Re_mks/r)^3*[
        5*(7*sinϕ^3 - 3*sinϕ)*x/r;
        5*(7*sinϕ^3 - 3*sinϕ)*y/r;
        3*(10*sinϕ^2 - 35/3*sinϕ^4-1)];
    
    g_j4 = -5/8*J4*(μ_mks/r^2)*(Re_mks/r)^4*[
        (3-42*sinϕ^2 + 63*sinϕ^4)*x/r;
        (3-42*sinϕ^2 + 63*sinϕ^4)*y/r;
       -(15-70*sinϕ^2 + 63*sinϕ^4)*z/r];
    
    g_j5 = -1/8*J5*(μ_mks/r^2)*(Re_mks/r)^5*[
        3*(35*sinϕ - 210*sinϕ^3 + 231*sinϕ^5)*x/r;
        3*(35*sinϕ - 210*sinϕ^3 + 231*sinϕ^5)*y/r;
       (15-315*sinϕ^2 + 945*sinϕ^4 - 693*sinϕ^6)];
    
    g_j6 = 1/16*J6*(μ_mks/r^2)*(Re_mks/r)^6*[
        (35 - 945*sinϕ^2 + 3465*sinϕ^4 - 3003*sinϕ^6)*x/r;
        (35 - 945*sinϕ^2 + 3465*sinϕ^4 - 3003*sinϕ^6)*y/r;
        (-315 + 2205*sinϕ^2 - 4851*sinϕ^4 + 3003*sinϕ^6)*z/r];
        
    g = g_kp + g_j2 #+ g_j3 + g_j4 + g_j5 + g_j6
    return g
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r1 = x[1:3]
    v1 = x[4:6]
    r2 = x[7:9]
    v2 = x[10:12]
    u1 = u[1]
    u2 = u[2]

    
    pos1_x = r1[1]
    pos1_y = r1[2]
    pos1_z = r1[3]
    r1_norm = sqrt(r1[1]^2+r1[2]^2+r1[3]^2)
    
    pos2_x = r2[1]
    pos2_y = r2[2]
    pos2_z = r2[3]
    r2_norm = sqrt(r2[1]^2+r2[2]^2+r2[3]^2)
    
    
    g1 = gravitational_acceleration(r1)
    g2 = gravitational_acceleration(r2)
    
    v1_rel = v1 - cross([0;0;Ωe_mks],r1) #Assume atmosphere co-rotates with the Earth
    ρ1 = density_mks(r1_norm - Re_mks)
    Fdrag1 = -0.5*(u1*AoM_mks)*Cd*ρ1*(norm(v1_rel))*v1_rel
    
    v2_rel = v2 - cross([0;0;Ωe_mks],r2) #Assume atmosphere co-rotates with the Earth
    ρ2 = density_mks(r2_norm - Re_mks)
    Fdrag2 = -0.5*(u2*AoM_mks)*Cd*ρ2*(norm(v2_rel))*v2_rel

    v̇1 = g1 + Fdrag1#- μ_mks*r1/(r1_norm^3) + Fdrag1 #g1
    v̇2 = g2 + Fdrag2#- μ_mks*r2/(r2_norm^3) + Fdrag2 #g2 
    return [v1; v̇1; v2; v̇2]
end


satellite_dynamics (generic function with 1 method)

In [12]:
#Initial and terminal constraints and #Inequality constraints
params = (
    l_12 = 10.0, # number of revolutions
    l_13 = 20.0,
    l_14 = 30.0,
    l_15 = 4.0,
    l_16 = 5.0,
    Δx0 = zeros(size(Ad,1)),
    Δaf = 0, #end at the same altitude
    u_min = 0.2,
    u_max = 1,
    Δa_min = -10/a_mks, # /num_sat
    Δa_max = 10/a_mks,) #/num_sat 

k4 = kvalues(a)[4]
params = (; params..., Δθf = 2*pi*[params.l_12,params.l_13,params.l_14,params.l_15,params.l_16],
)

params = (; params..., ΔΩf = k4*(params.Δθf))# Right Ascension Separation


(l_12 = 10.0, l_13 = 20.0, l_14 = 30.0, l_15 = 4.0, l_16 = 5.0, Δx0 = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Δaf = 0, u_min = 0.2, u_max = 1, Δa_min = -0.0014725371815638344, Δa_max = 0.0014725371815638344, Δθf = [62.83185307179586, 125.66370614359172, 188.49555921538757, 25.132741228718345, 31.41592653589793], ΔΩf = [-0.12123848743269434, -0.2424769748653887, -0.363715462298083, -0.04849539497307774, -0.06061924371634717])

In [13]:
#Discretized Optimal Control Problem Parameters
const Nx = size(Ad,1)     # number of state
const Nu = size(Bd,2)     # number of controls
const Tfinal = 15000#Int(round(800*P_mks)) # final time (number of orbits)
const Nt = Int(Tfinal/h)+1    # number of time steps
const Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));

#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]
U1 = [sparse(kron(Diagonal(I,Nt), sparse([I spzeros(1,(Nu-1+Nx))]))) for i = 1:num_sat] #Matrix that picks out all u of sat 1
U2 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,i) I spzeros(1,(Nu-i)+Nx-1)]))) for i = 1:num_sat] #Matrix that picks out all u of sat n
X1 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,Nu) spzeros(1,(3*(i-1))) I spzeros(1,3*(num_sat)-1-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x1 = Δθ
X2 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,Nu) spzeros(1,1) spzeros(1,(3*(i-1))) I spzeros(1,3*(num_sat)-2-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x2 = Δa
X3 = [sparse(kron(Diagonal(I,Nt), sparse([spzeros(1,Nu) spzeros(1,2) spzeros(1,(3*(i-1))) I spzeros(1,3*(num_sat)-3-3*(i-1))]))) for i = 1:num_sat] #Matrix that picks out all x3 = ΔΘ


X1F = spzeros(num_sat,(Nu+Nx)*Nt);  #Matrix that picks out x1f = Δθf
X2F = spzeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x2f = Δaf
# X3F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x3f = ΔΘf

for i = num_sat:-1:1
    X1F[num_sat-i+1,end-(i)*3+1] = 1.0
    X2F[num_sat-i+1,end-(i)*3+2] = 1.0
#     X3F[num_sat-i+1,end-(i)*3+3] = 1.0
end


In [14]:

# z = Variable(Nz)
b_c = zeros(Nt + Nt*num_sat + Nt*num_sat)
m_c = length(b_c)
z = Variable(Nz+m_c)
# t_c = Variable(m_c)

# linear cost term
c_c = [zeros(Nz);ones(m_c)]
function LP_controller(a,Δx, params)
    #Dynamics equality constraints C*z = d
    Ad, Bd = c2d(a, h, num_sat)
    C = sparse([[Bd -I spzeros(Nx,(Nt-1)*(Nu+Nx))]; spzeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [Ad Bd]) spzeros((Nt-1)*Nx,Nx)] + [spzeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[spzeros(Nx,Nu) Diagonal(-I,Nx)])]])
    d = sparse([-Ad*Δx; spzeros(Nx*(Nt-1))]);

    
    #Convex Formulation
    Ac = U1[1] 
    for i in 1:num_sat
        Ac  = [Ac; U2[i];]
    end

    for i in 1:num_sat
        Ac  = [Ac; X1[i]]
        b_c[Nt + Nt*num_sat + (i-1)*Nt+ 1:Nt + Nt*num_sat + (i)*Nt] .= params.Δθf[i]
    end
    
    # inequalities constraints
    G_c = [ Ac -I(m_c);
       -Ac -I(m_c)]

    h_c = [b_c; -b_c]
    problem = minimize(c_c'*z)
    problem.constraints += G_c*z <= h_c
    
#     problem = minimize(ones(m_c)'*t_c)
#     problem.constraints += t_c >= (Ac*z-b)
#     problem.constraints += t_c >= -(Ac*z-b)
    problem.constraints += [C*z[1:Nz] == d] #dynamics constraints

    for i = 1:num_sat
        problem.constraints += [X2F[i,:]'*z[1:Nz] == params.Δaf] #final altitude constraint
    #     problem.constraints += [X3F[i,:]'*z == ΔΩf] #final altitude constraint
        problem.constraints += [U1[i]*z[1:Nz] >= params.u_min; U1[i]*z[1:Nz] <= params.u_max] #drag constraints
        problem.constraints += [U2[i]*z[1:Nz] >= params.u_min; U2[i]*z[1:Nz] <= params.u_max] #drag constraints
        problem.constraints += [X2[i]*z[1:Nz] >= params.Δa_min; X2[i]*z[1:Nz] <= params.Δa_max]; #altitude constraints
    end
    
    solve!(problem, Mosek.Optimizer(), warmstart=true; silent_solver = true)#Mosek.Optimizer)
    # Optimizer
#     optimizer = GLPK.Optimizer()
#     MOI.set(optimizer, MOI.RawParameter("msg_lev"), GLPK.GLP_MSG_ON)
    #     solve!(problem,optimizer; silent_solver = false)
    
    u1 = zeros(size(U1[1])[1],num_sat) # sat 1 control
    u2 = zeros(size(U2[1])[1],num_sat) # sat n control
    for i = 1:num_sat
        u1[:,i] = U1[i]*z.value[1:Nz];
        u2[:,i] = U2[i]*z.value[1:Nz];
    end
    return [u1[1:h,:],u2[1:h,:]]
end

LP_controller (generic function with 1 method)

In [ ]:
using Statistics
# initial conditions
r_s0 = [a_mks; 0; 0]
v_s0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
r_s1 = [a_mks; 0; 0]
v_s1 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = repeat([r_s0; v_s0; r_s1; v_s1],num_sat);

# initialization results
# u1hist = zeros(size(U1[1])[1]*h,num_sat)
# u2hist = zeros(size(U2[1])[1]*h,num_sat)
# Δθhist = zeros(size(X1[1])[1]*h,num_sat)
# Δθunhist = zeros(size(X1[1])[1]*h,num_sat)
# Δahist = zeros(size(X2[1])[1]*h,num_sat)
# ΔΩhist = zeros(size(X3[1])[1]*h,num_sat)
# tp_hist = zeros(1,length(thist)*h)

u1hist = zeros(Tfinal,num_sat)
u2hist = zeros(Tfinal,num_sat)
Δθhist = zeros(Tfinal,num_sat)
Δθunhist = zeros(Tfinal,num_sat)
Δahist = zeros(Tfinal,num_sat)
ΔΩhist = zeros(Tfinal,num_sat)
tp_hist = zeros(1,Tfinal)

h_traj = 60 # integrate traj 1/5 Hz
T = P_mks#*h # integrate traj for n orbital period
# x_hist = zeros(length(x0),Int(round(T/h_traj))*Nt+1)
# t_hist = zeros(1,Int(round(T/h_traj))*Nt+1)
x_hist = zeros(length(x0),Int(round(T/h_traj))*Tfinal+1)
t_hist = zeros(1,Int(round(T/h_traj))*Tfinal+1)

# find first controller
xc = zeros(size(Ad,1))
for k = 0:num_sat-1
    xc[1+k*3:3+k*3,1] = cartesian_to_ΔθaΩ(mks_to_orbital(x0[1+k*12:6+k*12]),mks_to_orbital(x0[7+k*12:12+k*12]))
end
uk = LP_controller(a_vec, xc, params)

# store first step
x_hist[:,1] = x0
for k = 0:num_sat-1
    Δθhist[1,k+1] = xc[1+k*3]
    Δahist[1,k+1] = xc[2+k*3]
    ΔΩhist[1,k+1] = xc[3+k*3]
    u1hist[1,k+1] = uk[1][k+1]
    u2hist[1,k+1] = uk[2][k+1]
end

# initialize index
kk = 0 # traj steps count
mm = zeros(num_sat,1) # Wrapping matrix
ll = 0 # orbit counter
for k = 1:Tfinal-1 # control horizon
    ss = 0 # orbit counter for indexing
    for orbit = 1:1 # save results for each orbit
        ll += 1 # increase orbit counter
        for time = 0:h_traj:P_mks # loop for simulations for entire orbit
            kk += 1 # increase traj steps counter
            
            for jj = 0:num_sat-1 # loop for simulation for each couple of satellites
                u = [uk[1][jj*h+orbit],uk[2][jj*h+orbit]] # pick control stored
                x_hist[1+jj*12:12+jj*12,kk+1] .= sat_dynamics_rk4(x_hist[1+jj*12:12+jj*12,kk],u,h_traj) #simulate couple of sats
            end
            t_hist[kk+1] = time # save simulation time
        end
        
        for jj = 0:num_sat-1 # transform from cartesian to ΔθaΩ coordinates for each couple of sats
            xc[1+jj*3:3+jj*3,1] = cartesian_to_ΔθaΩ(mks_to_orbital(x_hist[1+jj*12:6+jj*12,kk+1]),mks_to_orbital(x_hist[7+jj*12:12+jj*12,kk+1]))
        end

        #save results
        for jj = 0:num_sat-1
            Δθhist[ll+1,jj+1] = xc[1+jj*3]
            Δahist[ll+1,jj+1] = xc[2+jj*3]
            ΔΩhist[ll+1,jj+1] = xc[3+jj*3]

            u1hist[ll+1,jj+1] = uk[1][jj*h+orbit]
            u2hist[ll+1,jj+1] = uk[2][jj*h+orbit]

#             println("sat num ", jj, " orbit ", orbit, " jj ", jj, " jj*h+orbit ",jj*h+orbit," u1 ",u1hist[ll+1,jj+1], " u2 ", u2hist[ll+1,jj+1])

            # angle wrapping, adjust angle for linearized dynamics of controller
            if abs(Δθhist[ll+1,jj+1]-Δθhist[ll,jj+1]) > pi-0.1
                mm[jj+1] =mm[jj+1]+ 2
                println("mm change ", abs(Δθhist[ll+1,jj+1]-Δθhist[ll,jj+1]), " - ",mm[jj+1] )
            end

            xc[1+jj*3] = mm[jj+1]*pi + xc[1+jj*3]
            Δθunhist[ll+1,jj+1] = xc[1+jj*3]
        end

        tp_hist[ll+1] = ll # store orbit time
        ss += 1*(num_sat-1) # increase index counter
#         println("End Loop - k ", k, " orbit ", orbit, " u = ",[uk[1][orbit],uk[2][orbit]])
    end
    
    # evaluate mean apoapsis between each couple of sats to update linearized dynamics
    a1 = zeros(num_sat,1); a2 = zeros(num_sat,1); a_upd = zeros(num_sat,1)
    for jj = 0:num_sat-1
        a1[jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[1+12*jj:6+12*jj,kk+1]))[2]
        a2[jj+1] = cartesian_to_θaΩ(mks_to_orbital(x_hist[7+12*jj:12+12*jj,kk+1]))[2]
        a_upd[jj+1] = mean([a1[jj+1],a2[jj+1]]) 
    end
    
    # evaluate controller
    uk = LP_controller(a_upd, xc, params)
#     println("k ",k, " a ", a_upd)#" u ", uk, "
    if k % 30 == 0
        println("k ",k, " - ",maximum(abs.(Δθunhist[ll+1,:].-params.Δθf[1:num_sat])), " - ", maximum(abs.(Δahist[ll+1,:].-params.Δaf[1])))
    end
    if maximum(abs.(Δθunhist[ll+1,:].-params.Δθf[1:num_sat]))<1e-4 && maximum(abs.(Δahist[ll+1,:].-params.Δaf[1]))<1e-4 # abs(Δθunhist[ll+1,:].-params.Δθf[1:num_sat])
        println("Reached Final Conditions")
        # Resize Results
        Δθhist = Δθhist[1:ll+1,:]
        Δahist = Δahist[1:ll+1,:]
        ΔΩhist = ΔΩhist[1:ll+1,:]
        u1hist = u1hist[1:ll+1,:]
        u2hist = u2hist[1:ll+1,:]
        tp_hist = tp_hist[1:ll+1]
        Δθunhist = Δθunhist[1:ll+1,:]

        break 
    end
end

x_hist = x_hist[:,1:kk]
t_hist = t_hist[:,1:kk]
print(kk)

In [ ]:
using Plots
plotlyjs()

mylabel = String[]
mylabel1 = String[]
mylabel2 = String[]
for i = 1:num_sat
    append!(mylabel,["Sat. 1-"*string(i+1)])
    append!(mylabel1,["Sat. 1"])
    append!(mylabel2,["Sat. "*string(i+1)])
end

pl = plot(tp_hist[1:end],u1hist[1:end,1],label=permutedims(mylabel1), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:best,legendfontsize=11)#,legend=:"best") #,ls=:dot
plot!(tp_hist[1:end],u2hist[1:end,:],label=permutedims(mylabel2), linewidth=2) #,ls=:dot
plot!(xlabel="Orbit Number", ylabel = "Control")#,legend=:"best", xtickfontsize=18, ytickfontsize=18)

savefig(pl,  "Plots/Control.tikz")    # produces a tikzpicture environment that can be included in other documents
savefig(pl, "Plots/Control.png")

In [ ]:
plot(t_hist[1:2000:end],x_hist[1,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[2,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[3,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[7,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[8,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(t_hist[1:2000:end],x_hist[9,1:2000:end])#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

In [ ]:
pl=plot(tp_hist[1:end],broadcast.(rad2deg,Δθunhist[1:end,:]),label=permutedims(mylabel), linewidth=2, xtickfontsize=11, ytickfontsize=11,yguidefontsize=14,xguidefontsize=14,legend=:topleft,legendfontsize=11)
plot!(xlabel="Orbit Number", ylabel = "True Anomaly Difference, deg")
# savefig(pl,  "Plots/Control.tikz")    # produces a tikzpicture environment that can be included in other documents
savefig(pl, "Plots/TrueAnomaly.png")

In [ ]:
plot(tp_hist[1:end],a_mks*Δahist[1:end,:], label=permutedims(mylabel))
plot!(xlabel="Orbit Number", ylabel = "Semi-major axis Difference, km")


In [ ]:
plot(tp_hist[1:end],broadcast.(rad2deg,ΔΩhist[1:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "RAAN Difference, deg",ylim = [-20,1])

In [ ]:
plot(tp_hist[1:end],a_mks*ΔΩhist[1:end,:],label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km",ylim = [-10*1e2,1*1e2 ])

In [ ]:
# Final RAAN
broadcast.(rad2deg,ΔΩhist[:,end]')

println("Targeted Δθf ",  broadcast.(rad2deg, params.Δθf[1:num_sat]), " deg" )
println("Reached Δθf ",  broadcast.(rad2deg, Δθunhist[end,:]), " deg" )


println("Targeted ΔΩ ",  broadcast.(rad2deg, params.ΔΩf[1:num_sat]), " deg" )
println("Reached ΔΩ ",  broadcast.(rad2deg, ΔΩhist[end,:]), " deg" )
println("Final Error ΔΩ ", broadcast.(rad2deg, (params.ΔΩf[1:num_sat] .- ΔΩhist[end,:])), " deg" )
println("Spherical distance ", a_mks*ΔΩhist[1:end,:], " km")
# println("Final Azimut Sat 1 ", broadcast.(rad2deg, ϕ_soln1[:,end]), " deg")
# println("Final Azimut Sat Coupled ", broadcast.(rad2deg, ϕ_soln2[:,end]), " deg")

# println("Final Elevation Sat 1 ", broadcast.(rad2deg, θ_soln1[:,end]), " deg")
# println("Final Elevation Sat Coupled ", broadcast.(rad2deg, θ_soln2[:,end]), " deg")

In [ ]:
# angle = acos((x_hist[1:3,end]*x_hist[7:9,end])/(norm(x_hist[1:3,end])*norm(x_hist[1:3,end])))
for i = 1:num_sat:
    println("Couple ",i ," Radius first spacecraft ", norm(x_hist[1:3,end])," - Radius second spacecraft ", norm(x_hist[7+(6*(i-1)):9(6*(i-1)),end]))
    distance = norm(x_hist[1:3,end])*acos(dot(x_hist[1:3,end],x_hist[7+(6*(i-1)):9+(6*(i-1)),end])/(norm(x_hist[1:3,end])*norm(x_hist[7+(6*(i-1)):9+(6*(i-1)),end])))
    println("Couple ",i ," Sperical Distance ", distance, " km")
end

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x_hist[1,1]/Re_mks-size_spacecraft[1]/2,x_hist[2,1]/Re_mks-size_spacecraft[2]/2,x_hist[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x_hist[1,1]/Re_mks-size_plate[1]/2,x_hist[2,1]/Re_mks-size_plate[2]/2,x_hist[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end


for j = 0 : num_sat-1
    if j == 0
        positions_1 = [x_hist[1,end]/Re_mks-origin_spacecraft[1],x_hist[2,end]/Re_mks-origin_spacecraft[2],x_hist[3,end]/Re_mks-origin_spacecraft[3]]
        position_1 = Translation(positions_1)
        name_sat = "satellite"*string(j+1)
        settransform!(vis[name_sat], position_1)
    end 
    positions_2 = [x_hist[7+(12*j),end]/Re_mks-origin_spacecraft[1],x_hist[8+(12*j),end]/Re_mks-origin_spacecraft[2],x_hist[9+(12*j),end]/Re_mks-origin_spacecraft[3]]
    position_2 = Translation(positions_2)
    name_sat = "satellite"*string(j+2)
    settransform!(vis[name_sat], position_2)
end



In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x_hist[1,1]/Re_mks-size_spacecraft[1]/2,x_hist[2,1]/Re_mks-size_spacecraft[2]/2,x_hist[3,1]/Re_mks-size_spacecraft[3]/2)
origin_plate = Vec(x_hist[1,1]/Re_mks-size_plate[1]/2,x_hist[2,1]/Re_mks-size_plate[2]/2,x_hist[3,1]/Re_mks-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

sleep(5)
for i = 1:10:length(x_hist[1,:])-1
    for j = 0 : num_sat-1
        if j == 0
#             positions_1 = [x0_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            positions_1 = [x_hist[1,i]/Re_mks-origin_spacecraft[1],x_hist[2,i]/Re_mks-origin_spacecraft[2],x_hist[3,i]/Re_mks-origin_spacecraft[3]]
#             rotations_1 = i*2*pi/length(x0_soln[1+(6*j-6),:])
            position_1 = Translation(positions_1)
#             rotation_1 = LinearMap(RotZ(rotations_1))
            name_sat = "satellite"*string(j+1)
            settransform!(vis[name_sat], position_1)
#             settransform!(vis[name_sat], compose(position_1, rotation_1))
        end 
#         positions_2 = [x1_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
        positions_2 = [x_hist[7+(12*j),i]/Re_mks-origin_spacecraft[1],x_hist[8+(12*j),i]/Re_mks-origin_spacecraft[2],x_hist[9+(12*j),i]/Re_mks-origin_spacecraft[3]]
#         rotations_2 = i*2*pi/length(x1_soln[1+(6*j-6),:])
        position_2 = Translation(positions_2)
#         rotation_2 = LinearMap(RotZ(rotations_2))
        name_sat = "satellite"*string(j+2)
        settransform!(vis[name_sat], position_2)
#         settransform!(vis[name_sat], compose(position_2, rotation_2))
    end
        #     settransform!(vis["plate1"], position)
    sleep(0.05)
end


In [ ]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    r_noJ2 = x[7:9]
    v_noJ2 = x[10:12]

    pos_x = r[1]
    pos_y = r[2]
    pos_z = r[3]
    r_norm = sqrt(r[1]^2+r[2]^2+r[3]^2)
    r_norm_noJ2 = sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2)
    
    g = +1.5*J2*μ_mks*Re_mks^2/r_norm^4*[
        pos_x/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_y/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_z/r_norm*(5*pos_z^2/r_norm^2-3)] - μ_mks*r/r_norm^3
    
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = density_mks(r_norm - Re_mks)
    Fdrag = -0.5*(u*AoM_mks)*Cd*ρ*(sqrt(v_rel[1]^2+v_rel[2]^2+v_rel[3]^2))*v_rel

    v_rel_noJ2 = v_noJ2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ_noJ2 = density_mks(r_norm_noJ2 - Re_mks)
    Fdrag_noJ2 = -0.5*(u*AoM_mks)*Cd*ρ_noJ2*(sqrt(v_rel_noJ2[1]^2+v_rel_noJ2[2]^2+v_rel_noJ2[3]^2))*v_rel_noJ2

    v̇ = g + Fdrag
    v̇_noJ2 = -μ_mks/(r_norm_noJ2)^3 * r_noJ2 +  Fdrag_noJ2
    return [v; v̇; v_noJ2; v̇_noJ2]
end


In [ ]:
#Initial conditions in Cartesian Coordinates
# r0 = [a; 0; 0]
# v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
# x0 = [r0; v0; r0; v0];

# r1 = [a; 0; 0]
# v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
# x1 = [r1; v1; r1; v1];

# tspan = (0.0, Tfinal+50)
# Nt_t = Tfinal+150

r0 = [a_mks; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ_mks/a_mks); sind(inc)*sqrt(μ_mks/a_mks)]
x0 = [r0; v0; r0; v0];

r1 = [a_mks; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ_mks/(a_mks)); sind(inc)*sqrt(μ_mks/(a_mks))]
x1 = [r1; v1; r1; v1];

step_t = 60*10 # save results every 10 mins
tspan = (0.0, Int(round((Tfinal)*P_mks/step_t)*step_t))
Nt_t = Int(round((Tfinal)*P_mks/step_t))-1#/100

In [ ]:
using Interpolations
#Full Orbit Mechanics Simulation and save results
# Nt = 150
# Nt = Nt*3
temp = LinRange(tspan[1],tspan[2],Nt_t)
x0_soln = zeros(6*num_sat,Nt_t)
x1_soln = zeros(6*num_sat,Nt_t)

x0_noJ2_soln = zeros(6*num_sat,Nt_t)
x1_noJ2_soln = zeros(6*num_sat,Nt_t)

Δa_soln = zeros(num_sat,Nt_t)
Δθ_soln = zeros(num_sat,Nt_t)
ΔΩ_soln = zeros(num_sat,Nt_t)

r_soln1 = zeros(num_sat,Nt_t)
ϕ_soln1 = zeros(num_sat,Nt_t)
θ_soln1 = zeros(num_sat,Nt_t)
u_soln1 = zeros(num_sat,Nt_t)

r_soln2 = zeros(num_sat,Nt_t)
ϕ_soln2 = zeros(num_sat,Nt_t)
θ_soln2 = zeros(num_sat,Nt_t)
u_soln2 = zeros(num_sat,Nt_t)

a_soln1 = zeros(num_sat,Nt_t)
ν_soln1 = zeros(num_sat,Nt_t)
Ω_soln1 = zeros(num_sat,Nt_t)

a_soln2 = zeros(num_sat,Nt_t)
ν_soln2 = zeros(num_sat,Nt_t)
Ω_soln2 = zeros(num_sat,Nt_t)

r_soln1_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln1_noJ2 = zeros(num_sat,Nt_t)
θ_soln1_noJ2 = zeros(num_sat,Nt_t)
u_soln1_noJ2 = zeros(num_sat,Nt_t)

r_soln2_noJ2 = zeros(num_sat,Nt_t)
ϕ_soln2_noJ2 = zeros(num_sat,Nt_t)
θ_soln2_noJ2 = zeros(num_sat,Nt_t)
u_soln2_noJ2 = zeros(num_sat,Nt_t)

h_traj = 1#step_t#/P_mks
thist_c = thist.*P_mks
for i = 1:num_sat
    u1interp = 0;#LinearInterpolation(thist_c, u1hist[:,i])
    u2interp = 1;#LinearInterpolation(thist_c, u2hist[:,i])

    x_hist1 = zeros(length(x0),Int(tspan[2]/h_traj))
    x_hist2 = zeros(length(x1),Int(tspan[2]/h_traj))
    x_hist1[:,1] = x0 
    x_hist2[:,1] = x1 
    temp = LinRange(tspan[1],tspan[2],Nt_t)
    k = 0 
    save_index = 0
    for t = tspan[1]:h_traj:tspan[2]-step_t# temp# 1:100:tspan[2]-1
        k += 1
        #Stored Controller
        if t < tspan[2]
            u1 = 0;#u1interp(t) 
            u2 = 1;#u2interp(t) 
        else
            u1 = 0;#u_soln1[i,k]
            u2 = 0;#u_soln2[i,k]
        end

        x_hist1[:,k+1] .= sat_dynamics_rk4(x_hist1[:,k],u1,h_traj)
        x_hist2[:,k+1] .= sat_dynamics_rk4(x_hist2[:,k],u2,h_traj)

        # save results
        if k%step_t == 0
            save_index += 1
            u_soln1[i,save_index] = u1
            u_soln2[i,save_index] = u2
            x0_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[1:6,k]
            x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist1[7:12,k]
        
            x1_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[1:6,k]
            x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index] = x_hist2[7:12,k]
    
            rϕθ1_noJ2 = cartesian_to_spherical(x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2_noJ2 = cartesian_to_spherical(x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ1 = cartesian_to_spherical(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            rϕθ2 = cartesian_to_spherical(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index])
            ΔaθΩ = cartesian_to_ΔθaΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]), mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ1 = cartesian_to_θaΩ(mks_to_orbital(x0_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
            aθΩ2 = cartesian_to_θaΩ(mks_to_orbital(x1_soln[6*(i-1)+1:6*(i-1)+6,save_index]))
    
            Δθ_soln[i,save_index] = ΔaθΩ[1]    
            Δa_soln[i,save_index] = ΔaθΩ[2]
            ΔΩ_soln[i,save_index] = ΔaθΩ[3]

            ν_soln1[i,save_index] = aθΩ1[1]
            a_soln1[i,save_index] = aθΩ1[2]
            Ω_soln1[i,save_index] = aθΩ1[3]
        
            ν_soln2[i,save_index] = aθΩ2[1]
            a_soln2[i,save_index] = aθΩ2[2]
            Ω_soln2[i,save_index] = aθΩ2[3]
        
            r_soln1_noJ2[i,save_index] = rϕθ1_noJ2[1]
            ϕ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[2]
            θ_soln1_noJ2[i,save_index] = rϕθ1_noJ2[3]
    
            r_soln2_noJ2[i,save_index] = rϕθ2_noJ2[1]
            ϕ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[2]
            θ_soln2_noJ2[i,save_index] = rϕθ2_noJ2[3]
        
            r_soln1[i,save_index] = rϕθ1[1]
            ϕ_soln1[i,save_index] = rϕθ1[2]
            θ_soln1[i,save_index] = rϕθ1[3]
    
            r_soln2[i,save_index] = rϕθ2[1]
            ϕ_soln2[i,save_index] = rϕθ2[2]
            θ_soln2[i,save_index] = rϕθ2[3]
        end

    end
end
tsamp = LinRange(tspan[1],tspan[2],Nt_t);


In [ ]:
using Plots
# tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp[1:30:end]./(60*60*24),x0_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x0_soln[3,1:30:end],color=:green)

plot!(tsamp[1:30:end]./(60*60*24),x1_soln[1,1:30:end],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[2,1:30:end],color=:blue)
plot!(tsamp[1:30:end]./(60*60*24),x1_soln[3,1:30:end],color=:green)

# plot(tsamp,x0_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x0_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x0_noJ2_soln[3,:],color=:green)

# plot!(tsamp,x1_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x1_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x1_noJ2_soln[3,:],color=:green)
# x1_soln[3,:]

In [ ]:
plot(tsamp[1:30:end]./(60*60*24),broadcast.(rad2deg,ν_soln1[1,1:30:end]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),broadcast.(rad2deg,Ω_soln2[1,:]),color=:blue)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

# length(Ω_soln1[1,:][Ω_soln1[1,:] .>3.])

In [ ]:
tsamp = LinRange(tspan[1],tspan[2],Nt_t)
plot(tsamp[1:30:end]./(60*60*24),Δa_soln[1:30:end]',label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "Altitude Difference, km")


In [ ]:
plot(tsamp./(60*60*24),broadcast.(rad2deg,Δθ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "True Anomaly Difference, deg")

In [ ]:
plot(tsamp./(60*40*24),broadcast.(rad2deg,ΔΩ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, days", ylabel = "RAAN Difference, deg", ylim = [-10,5])

In [ ]:
plot(tsamp,a_mks*ΔΩ_soln',label=permutedims(mylabel), ylim = [-100*1e1,10*1e2])
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [ ]:
plot(tsamp./(60*60*24),u_soln1',color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*60*24),u_soln2',label=permutedims(mylabel2))
plot!(xlabel="Time, days", ylabel = "Control")


In [ ]:
plot(tsamp./(60*40*24),r_soln1_noJ2'.-Re_mks,color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp./(60*40*24),r_soln2_noJ2'.-Re_mks,label=permutedims(mylabel2))
for i=1:num_sat
    println(r_soln1_noJ2[1]-Re_mks," ",params.Δa_max*a_mks*i)
    plot!([tsamp[1]./(60*40*24), tsamp[end]./(60*40*24)],[r_soln1_noJ2[1]-Re_mks-params.Δa_max*a_mks*i, r_soln1_noJ2[1]-Re_mks[1]-params.Δa_max*a_mks*i],color=:red, linewidth=2,label="Limit")
end
plot!(xlabel= "Time, days",ylabel= "Altitude, km")


In [ ]:
plot(tsamp[1:30:end]./(60*40*24),ϕ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),ϕ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Azimuth, rad")

In [ ]:
plot(tsamp[1:30:end]./(60*40*24),θ_soln1_noJ2[:,1:30:end]',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp[1:30:end]./(60*40*24),θ_soln2_noJ2[:,1:30:end]',label=permutedims(mylabel2))
plot!(xlabel= "Time, days",ylabel= "Elevation, rad")

In [ ]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [ ]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [ ]:
k1/k2

In [ ]:
(k1/k2)*2*pi

In [ ]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi